# ICS 3202: Artificial Intelligence
## Practical Exercise 4: Developing a Knowledge Based Agent

Class: ICS_3A

| No. | Name | Admission No. |
|-----|------|---------------|
| 1 | Seyianoi Sankok | 190384 |
| 2 | Annie Wanjohi | 171001 |

## Introduction

In Practical Exercise 3 the trailer only knew the towns and the distances between them. It did not know about traffic rules, road conditions or the weather. Here it becomes a knowledge based agent, with the three parts we saw in class.

| Part | In this notebook |
|------|------------------|
| Knowledge base | Facts and IF THEN rules about the roads |
| Inference engine | Forward chaining, which applies the rules to the facts to get new facts |
| Interface | Typed commands that TELL the agent something or ASK it something |

In every step the agent TELLs the knowledge base what it perceives, ASKs it what to do, and then TELLs it the action it chose.

The knowledge comes from these places.

| Knowledge | Source |
|-----------|--------|
| Towns, roads and distances | Practical Exercise 3 |
| Road numbers (A, B, C) and tolls | Google Maps directions |
| Speed limits | The class example of 80 km/h on highways. The 60 km/h speed on C roads, steep roads and in heavy rain is a safety rule chosen by the group |
| Height of each town | Approximate, in metres |
| Weather | Typed in by the user |

A road is steep when the two towns it joins differ in height by more than 400 m.

## 1. The data

In [1]:
# Roads from Practical Exercise 3: town A, town B, distance in km, road number
ROADS = [
    ("Nairobi", "Thika", 45.1, "A2"),
    ("Nairobi", "Naivasha", 90.4, "A8"),
    ("Nairobi", "Narok", 143, "B3"),
    ("Thika", "Nyeri", 108, "A2"),
    ("Naivasha", "Nakuru", 69.6, "A8"),
    ("Nyeri", "Nakuru", 167, "B5"),
    ("Narok", "Bomet", 81.0, "B3"),
    ("Nakuru", "Molo", 47.7, "A8"),
    ("Nakuru", "Eldoret", 156, "A8"),
    ("Molo", "Kericho", 64.2, "B1"),
    ("Bomet", "Kericho", 72.1, "C23"),
    ("Bomet", "Kisii", 82.4, "B3"),
    ("Kericho", "Kisumu", 84.0, "B1"),
    ("Kisii", "Kisumu", 107, "A1"),
    ("Eldoret", "Kakamega", 87.9, "A1"),
    ("Eldoret", "Kisumu", 126, "C39"),
    ("Kakamega", "Kisumu", 53.4, "A1"),
]

# Google Maps showed "This route has tolls" for these two roads
TOLLS = [("Nairobi", "Naivasha"), ("Nairobi", "Narok")]

# height of each town in metres (approximate)
ELEVATION = {
    "Nairobi": 1795, "Thika": 1631, "Nyeri": 1759, "Naivasha": 1884,
    "Nakuru": 1850, "Molo": 2400, "Kericho": 2002, "Eldoret": 2100,
    "Narok": 1827, "Bomet": 1990, "Kisii": 1700, "Kakamega": 1535,
    "Kisumu": 1131,
}

## 2. The knowledge base and the inference engine

Facts are stored as tuples, for example ("road", "Nairobi", "Thika", 45.1). A rule has a name, some conditions and a conclusion. A word that starts with a question mark is a variable.

The infer method is forward chaining. It applies every rule to the facts, adds the new facts, and repeats until nothing new appears.

In [2]:
def match(pattern, fact, binding):
    # try to fit a pattern like ("road", "?a", "?b", "?d") onto one fact
    if len(pattern) != len(fact):
        return None
    b = dict(binding)
    for p, f in zip(pattern, fact):
        if isinstance(p, str) and p.startswith("?"):
            if p in b and b[p] != f:
                return None
            b[p] = f
        elif p != f:
            return None
    return b


class KnowledgeBase:
    def __init__(self):
        self.facts = set()      # facts the agent was told
        self.rules = []         # IF THEN rules
        self.derived = set()    # facts the engine worked out
        self.dirty = True

    def tell(self, fact):
        self.facts.add(fact)
        self.dirty = True

    def untell(self, pattern):
        for f in list(self.facts):
            if match(pattern, f, {}) is not None:
                self.facts.remove(f)
                self.dirty = True

    def add_rule(self, name, conditions, conclusion):
        self.rules.append((name, conditions, conclusion))
        self.dirty = True

    def solve(self, conditions, facts, binding):
        # find every way the conditions of a rule can be true
        if not conditions:
            yield binding
            return
        first, rest = conditions[0], conditions[1:]
        if first[0] == "not":
            if not any(match(first[1], f, binding) is not None for f in facts):
                yield from self.solve(rest, facts, binding)
        elif first[0] == "gt":
            if binding[first[1]] > first[2]:
                yield from self.solve(rest, facts, binding)
        else:
            for f in facts:
                b = match(first, f, binding)
                if b is not None:
                    yield from self.solve(rest, facts, b)

    def infer(self):
        # forward chaining: apply the rules to the facts, add what is new, repeat until nothing changes
        self.derived = set()
        changed = True
        while changed:
            changed = False
            facts = self.facts | self.derived
            for name, conditions, conclusion in self.rules:
                for b in self.solve(conditions, facts, {}):
                    new = tuple(b.get(x, x) for x in conclusion)
                    if new not in facts:
                        self.derived.add(new)
                        changed = True
        self.dirty = False

    def ask(self, pattern):
        if self.dirty:
            self.infer()
        found = [f for f in self.facts | self.derived if match(pattern, f, {}) is not None]
        return sorted(found, key=str)

    def safe_speed(self, a, b):
        # several rules can give a speed, the lowest one is the safe one
        return min(f[3] for f in self.ask(("safe_speed", a, b, "?v")))

    def distance(self, a, b):
        return self.ask(("road", a, b, "?d"))[0][3]

## 3. Facts and rules

This puts the roads into the knowledge base as facts and adds the IF THEN rules.

In [3]:
def build_knowledge_base():
    kb = KnowledgeBase()

    # facts
    for a, b, d, number in ROADS:
        for x, y in ((a, b), (b, a)):
            kb.tell(("road", x, y, d))
            kb.tell(("road_class", x, y, number[0]))
            kb.tell(("height_change", x, y, abs(ELEVATION[x] - ELEVATION[y])))
    for a, b in TOLLS:
        kb.tell(("toll", a, b))
        kb.tell(("toll", b, a))

    # rules
    kb.add_rule("R1 A roads", [("road_class", "?a", "?b", "A")], ("speed_limit", "?a", "?b", 80))
    kb.add_rule("R2 B roads", [("road_class", "?a", "?b", "B")], ("speed_limit", "?a", "?b", 80))
    kb.add_rule("R3 C roads", [("road_class", "?a", "?b", "C")], ("speed_limit", "?a", "?b", 60))
    kb.add_rule("R4 steep road", [("height_change", "?a", "?b", "?h"), ("gt", "?h", 400)], ("steep", "?a", "?b"))
    kb.add_rule("R5 wet road, rain at the start", [("road", "?a", "?b", "?d"), ("weather", "?a", "heavy_rain")], ("wet", "?a", "?b"))
    kb.add_rule("R6 wet road, rain at the end", [("road", "?a", "?b", "?d"), ("weather", "?b", "heavy_rain")], ("wet", "?a", "?b"))
    kb.add_rule("R7 normal safe speed", [("speed_limit", "?a", "?b", "?v")], ("safe_speed", "?a", "?b", "?v"))
    kb.add_rule("R8 wet road is slower", [("wet", "?a", "?b")], ("safe_speed", "?a", "?b", 60))
    kb.add_rule("R9 steep road is slower", [("steep", "?a", "?b")], ("safe_speed", "?a", "?b", 60))
    kb.add_rule("R10 open road", [("road", "?a", "?b", "?d"), ("not", ("closed", "?a", "?b"))], ("usable", "?a", "?b", "?d"))
    kb.add_rule("R11 toll advice", [("toll", "?a", "?b")], ("advice", "?a", "?b", "this road has tolls, carry the toll fee"))
    kb.add_rule("R12 steep advice", [("steep", "?a", "?b")], ("advice", "?a", "?b", "steep road, use a low gear going down"))
    kb.add_rule("R13 wet advice", [("wet", "?a", "?b")], ("advice", "?a", "?b", "wet road, keep a long gap and brake gently"))
    return kb


KB = build_knowledge_base()
KB.infer()
print("Facts told:", len(KB.facts))
print("Facts worked out by the rules:", len(KB.derived))
print("Rules:", len(KB.rules))

Facts told: 106
Facts worked out by the rules: 140
Rules: 13


## 4. The route planner

This is the uniform cost search from Practical Exercise 3. The difference is that the roads now come from the knowledge base, so a closed road is left out of the search.

In [4]:
def plan_route(kb, start, goal, by="distance"):
    # uniform cost search as in Practical Exercise 3, but the roads come from the knowledge base
    speeds = {}
    for f in kb.ask(("safe_speed", "?a", "?b", "?v")):
        speeds[(f[1], f[2])] = min(f[3], speeds.get((f[1], f[2]), 999))
    graph = {}
    for f in kb.ask(("usable", "?a", "?b", "?d")):
        cost = f[3] if by == "distance" else f[3] / speeds[(f[1], f[2])]
        graph.setdefault(f[1], []).append((f[2], cost))

    frontier = [(0.0, [start])]
    explored = []
    while frontier:
        frontier.sort(key=lambda item: item[0])
        cost, path = frontier.pop(0)
        town = path[-1]
        if town == goal:
            return round(cost, 1 if by == "distance" else 3), path
        if town in explored:
            continue
        explored.append(town)
        for nxt, c in graph.get(town, []):
            if nxt not in explored:
                frontier.append((cost + c, path + [nxt]))
    return None


def hours_and_minutes(hours):
    minutes = round(hours * 60)
    return f"{minutes // 60} h {minutes % 60:02d} min"

## 5. The agent

The agent drives from town to town. In every step it TELLs the percept, ASKs for the next action and TELLs the action it chose. It plans again at every town, so it reacts when the knowledge changes.

In [5]:
def drive(kb, start, goal):
    # the agent drives from town to town: TELL the percept, ASK for the action, TELL the action chosen
    town, step, km, hours = start, 0, 0.0, 0.0
    kb.untell(("goal", "?g"))
    kb.tell(("goal", goal))
    print(f"TELL goal: reach {goal} from {start}\n")

    while town != goal and step < 25:
        step += 1
        weather = kb.ask(("weather", town, "?w"))
        weather = weather[0][2] if weather else "clear"
        kb.untell(("at", "?t"))
        kb.tell(("at", town))
        print(f"Step {step}")
        print(f"  TELL percept: in {town}, weather is {weather}")

        route = plan_route(kb, town, goal)
        if route is None:
            print(f"  ASK: no open road leads to {goal}, so the trailer waits in {town}")
            return
        cost, path = route
        nxt = path[1]
        speed = kb.safe_speed(town, nxt)
        d = kb.distance(town, nxt)
        print(f"  ASK action: drive to {nxt}, {d} km at {speed} km/h")
        print(f"  Route from here: {' > '.join(path)}")
        for f in kb.ask(("advice", town, nxt, "?text")):
            print(f"  Advice: {f[3]}")
        kb.tell(("did", step, town, nxt, speed))
        print(f"  TELL action chosen: drive {town} to {nxt}\n")
        km += d
        hours += d / speed
        town = nxt

    if town == goal:
        print(f"Arrived in {goal} after {step} steps")
        print(f"Distance driven: {km:.1f} km")
        print(f"Driving time at the safe speeds: {hours_and_minutes(hours)}")

## 6. The interface

The user types short commands. Close, open and rain TELL the agent something new. Route, speed and drive ASK it something. Type help to see the list.

In [6]:
HELP = """Commands:
  route A B         cheapest route by distance, e.g. route Nairobi Kisumu
  route A B time    fastest route at the safe speeds
  drive A B         drive the trip step by step
  speed A B         safe speed on the road from A to B
  close A B         TELL the agent the road is closed
  open A B          TELL the agent the road is open again
  rain T            TELL the agent it is raining heavily in town T
  rain T clear      TELL the agent the rain has stopped
  help              show this list"""


def handle(command, kb):
    words = command.strip().split()
    if not words:
        return
    word = words[0].lower()
    rest = [w.lower() for w in words[1:]]
    towns = [w.capitalize() for w in words[1:] if w.capitalize() in ELEVATION]

    if word == "help":
        print(HELP)
    elif word == "rain" and towns:
        kb.untell(("weather", towns[0], "?w"))
        if "clear" in rest:
            print(f"TELL: the rain in {towns[0]} has stopped")
        else:
            kb.tell(("weather", towns[0], "heavy_rain"))
            print(f"TELL: it is raining heavily in {towns[0]}")
    elif word in ("route", "drive", "speed", "close", "open"):
        if len(towns) < 2:
            print("I need two towns. Towns I know:", ", ".join(ELEVATION))
            return
        a, b = towns[0], towns[1]
        if word == "drive":
            drive(kb, a, b)
        elif word == "route":
            by = "time" if "time" in rest else "distance"
            found = plan_route(kb, a, b, by)
            if found is None:
                print(f"There is no open road from {a} to {b}")
            else:
                cost, path = found
                print("Route:", " > ".join(path))
                print(f"Distance: {cost} km" if by == "distance" else f"Driving time: {hours_and_minutes(cost)}")
        elif not kb.ask(("road", a, b, "?d")):
            print(f"There is no direct road between {a} and {b}")
        elif word == "speed":
            print(f"Safe speed from {a} to {b}: {kb.safe_speed(a, b)} km/h")
        elif word == "close":
            kb.tell(("closed", a, b))
            kb.tell(("closed", b, a))
            print(f"TELL: the road between {a} and {b} is closed")
        else:
            kb.untell(("closed", a, b))
            kb.untell(("closed", b, a))
            print(f"TELL: the road between {a} and {b} is open")
    else:
        print("I did not understand that. Type help to see the commands.")


def chat(kb):
    print("Trailer agent. Type help for the commands and quit to stop.")
    while True:
        command = input("> ")
        if command.strip().lower() in ("quit", "exit"):
            print("Goodbye")
            break
        handle(command, kb)

## 7. Demonstration

In [7]:
commands = [
    "speed Nairobi Naivasha",
    "speed Nakuru Molo",
    "route Nairobi Kisumu",
    "route Nairobi Kisumu time",
    "rain Molo",
    "route Nairobi Kisumu time",
    "rain Molo clear",
    "close Naivasha Nakuru",
    "route Nairobi Kisumu",
    "open Naivasha Nakuru",
]
for command in commands:
    print("> " + command)
    handle(command, KB)
    print()

> speed Nairobi Naivasha
Safe speed from Nairobi to Naivasha: 80 km/h

> speed Nakuru Molo
Safe speed from Nakuru to Molo: 60 km/h

> route Nairobi Kisumu
Route: Nairobi > Naivasha > Nakuru > Molo > Kericho > Kisumu
Distance: 355.9 km

> route Nairobi Kisumu time
Route: Nairobi > Naivasha > Nakuru > Molo > Kericho > Kisumu
Driving time: 5 h 00 min

> rain Molo
TELL: it is raining heavily in Molo

> route Nairobi Kisumu time
Route: Nairobi > Naivasha > Nakuru > Molo > Kericho > Kisumu
Driving time: 5 h 16 min

> rain Molo clear
TELL: the rain in Molo has stopped

> close Naivasha Nakuru
TELL: the road between Naivasha and Nakuru is closed

> route Nairobi Kisumu
Route: Nairobi > Narok > Bomet > Kericho > Kisumu
Distance: 380.1 km

> open Naivasha Nakuru
TELL: the road between Naivasha and Nakuru is open



In [8]:
drive(KB, "Nairobi", "Kisumu")

TELL goal: reach Kisumu from Nairobi

Step 1
  TELL percept: in Nairobi, weather is clear
  ASK action: drive to Naivasha, 90.4 km at 80 km/h
  Route from here: Nairobi > Naivasha > Nakuru > Molo > Kericho > Kisumu
  Advice: this road has tolls, carry the toll fee
  TELL action chosen: drive Nairobi to Naivasha

Step 2
  TELL percept: in Naivasha, weather is clear
  ASK action: drive to Nakuru, 69.6 km at 80 km/h
  Route from here: Naivasha > Nakuru > Molo > Kericho > Kisumu
  TELL action chosen: drive Naivasha to Nakuru

Step 3
  TELL percept: in Nakuru, weather is clear
  ASK action: drive to Molo, 47.7 km at 60 km/h
  Route from here: Nakuru > Molo > Kericho > Kisumu
  Advice: steep road, use a low gear going down
  TELL action chosen: drive Nakuru to Molo

Step 4
  TELL percept: in Molo, weather is clear
  ASK action: drive to Kericho, 64.2 km at 80 km/h
  Route from here: Molo > Kericho > Kisumu
  TELL action chosen: drive Molo to Kericho

Step 5
  TELL percept: in Kericho, weathe

## 8. Result

With all the roads open the best route is Nairobi, Naivasha, Nakuru, Molo, Kericho, Kisumu. The cost is 355.9 km, the same answer as Practical Exercise 3.

When the road between Naivasha and Nakuru is closed, the agent updates its knowledge and the route changes to Nairobi, Narok, Bomet, Kericho, Kisumu at 380.1 km. This is the next best route from Practical Exercise 3.

Because of the extra knowledge, the agent also gives a driving time, warns about tolls and slows down on steep roads and in heavy rain.

## 9. Try it

Run the next cell and type a command after the arrow. For example, route Nairobi Kisumu, then close Naivasha Nakuru and ask for the route again. Type quit to stop.

In [ ]:
chat(KB)